# Model Comparison

Run this after notebooks 02 and 05. It compares the wav2vec2 main model against the CNN baseline and saves report-ready tables and plots.

In [ ]:
from __future__ import annotations

import importlib.util
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/pavannn16/speech-emotion-directions.git"
REPO_NAME = "speech-emotion-directions"

os.environ["TOKENIZERS_PARALLELISM"] = "false"


def running_in_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except ImportError:
        return False


IS_COLAB = running_in_colab()
print("Running in Colab:", IS_COLAB)


def run_command(cmd: list[str], cwd: Path | None = None) -> None:
    print("+", " ".join(cmd))
    subprocess.check_call(cmd, cwd=str(cwd) if cwd else None)



def ensure_packages() -> None:
    package_map = {
        "yaml": "pyyaml",
        "pandas": "pandas",
        "numpy": "numpy",
        "matplotlib": "matplotlib",
        "seaborn": "seaborn",
        "umap": "umap-learn",
        "torch": "torch",
        "transformers": "transformers",
        "datasets": "datasets",
        "accelerate": "accelerate",
        "sklearn": "scikit-learn",
        "tqdm": "tqdm",
        "huggingface_hub": "huggingface_hub",
    }
    missing = sorted({pkg for module, pkg in package_map.items() if importlib.util.find_spec(module) is None})
    if missing:
        run_command([sys.executable, "-m", "pip", "install", "-q", *missing])
    else:
        print("Required analysis packages are already available.")



def find_local_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, cwd.parent, cwd / "FinalProject"]:
        candidate = candidate.resolve()
        if (candidate / "configs" / "wav2vec.yaml").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Could not find the project root locally.")



def repo_status_lines(repo_root: Path) -> list[str]:
    result = subprocess.run(
        ["git", "-C", str(repo_root), "status", "--short"],
        text=True,
        capture_output=True,
        check=True,
    )
    return [line for line in result.stdout.splitlines() if line.strip()]



def repo_ahead_behind(repo_root: Path) -> tuple[int, int]:
    result = subprocess.run(
        ["git", "-C", str(repo_root), "rev-list", "--left-right", "--count", "HEAD...origin/main"],
        text=True,
        capture_output=True,
        check=False,
    )
    if result.returncode != 0 or not result.stdout.strip():
        return 0, 0
    ahead_str, behind_str = result.stdout.strip().split()
    return int(ahead_str), int(behind_str)



def clone_clean_code_checkout(clean_root: Path) -> Path:
    if clean_root.exists():
        shutil.rmtree(clean_root)
    run_command(["git", "clone", "--depth", "1", REPO_URL, str(clean_root)])
    return clean_root



def prepare_project_and_code_roots() -> tuple[Path, Path]:
    runtime_root = Path("/content") / REPO_NAME
    clean_root = Path("/content") / f"{REPO_NAME}-clean"

    if runtime_root.exists() and (runtime_root / ".git").exists():
        try:
            run_command(["git", "-C", str(runtime_root), "fetch", "origin"])
        except subprocess.CalledProcessError as exc:
            print(f"Fetch failed for existing Colab repo; continuing with local state. Details: {exc}")

        status_lines = repo_status_lines(runtime_root)
        ahead, behind = repo_ahead_behind(runtime_root)

        if not status_lines and ahead == 0:
            if behind > 0:
                try:
                    run_command(["git", "-C", str(runtime_root), "pull", "--ff-only", "origin", "main"])
                    code_root = runtime_root
                except subprocess.CalledProcessError as exc:
                    print(f"Fast-forward pull failed; using a clean code checkout instead. Details: {exc}")
                    code_root = clone_clean_code_checkout(clean_root)
            else:
                code_root = runtime_root
        else:
            print(f"Using a clean code checkout because {runtime_root} has local changes or local commits.")
            for line in status_lines[:10]:
                print(" -", line)
            if ahead or behind:
                print(f"Repo divergence relative to origin/main: ahead={ahead}, behind={behind}")
            code_root = clone_clean_code_checkout(clean_root)

        project_root = runtime_root
    elif runtime_root.exists():
        print(f"Using existing non-git project directory for data/artifacts: {runtime_root}")
        project_root = runtime_root
        code_root = clone_clean_code_checkout(clean_root)
    else:
        run_command(["git", "clone", "--depth", "1", REPO_URL, str(runtime_root)])
        project_root = runtime_root
        code_root = runtime_root

    return project_root, code_root


ensure_packages()

if IS_COLAB:
    from google.colab import drive  # type: ignore

    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT, CODE_ROOT = prepare_project_and_code_roots()
else:
    PROJECT_ROOT = find_local_project_root()
    CODE_ROOT = PROJECT_ROOT

def module_uses_code_root(module_file: str | None, code_root: Path) -> bool:
    if module_file is None:
        return False
    try:
        Path(module_file).resolve().relative_to(code_root.resolve())
        return True
    except ValueError:
        return False


os.chdir(PROJECT_ROOT)
for root in [str(CODE_ROOT), str(PROJECT_ROOT)]:
    while root in sys.path:
        sys.path.remove(root)

sys.path.insert(0, str(CODE_ROOT))
if str(PROJECT_ROOT) != str(CODE_ROOT):
    sys.path.insert(1, str(PROJECT_ROOT))

src_module = sys.modules.get("src")
src_file = getattr(src_module, "__file__", None)
if src_module is not None and not module_uses_code_root(src_file, CODE_ROOT):
    stale_modules = [name for name in list(sys.modules) if name == "src" or name.startswith("src.")]
    for name in stale_modules:
        del sys.modules[name]

print("Project root:", PROJECT_ROOT)
print("Code root:", CODE_ROOT)
print("Working directory:", Path.cwd().resolve())

In [ ]:
from pathlib import Path

wav_experiment_name = 'wav2vec_ravdess_speaker_independent_v1'
cnn_experiment_name = 'cnn_baseline_ravdess_speaker_independent_v1'
comparison_name = 'wav2vec_vs_cnn_baseline'

local_wav_checkpoint_dir = PROJECT_ROOT / 'artifacts' / 'checkpoints' / wav_experiment_name
local_cnn_checkpoint_dir = PROJECT_ROOT / 'artifacts' / 'checkpoints' / cnn_experiment_name
local_comparison_dir = PROJECT_ROOT / 'artifacts' / 'reports' / comparison_name
local_comparison_dir.mkdir(parents=True, exist_ok=True)

drive_root = Path('/content/drive/MyDrive') / 'speech-emotion-directions' / 'runs' if IS_COLAB else None
drive_wav_checkpoint_dir = drive_root / wav_experiment_name / 'checkpoint' if drive_root is not None else None
drive_cnn_checkpoint_dir = drive_root / cnn_experiment_name / 'checkpoint' if drive_root is not None else None
drive_comparison_dir = drive_root / comparison_name / 'comparison_report' if drive_root is not None else None

wav_checkpoint_dir = local_wav_checkpoint_dir
if not (wav_checkpoint_dir / 'test_metrics.json').exists() and drive_wav_checkpoint_dir is not None and (drive_wav_checkpoint_dir / 'test_metrics.json').exists():
    wav_checkpoint_dir = drive_wav_checkpoint_dir

cnn_checkpoint_dir = local_cnn_checkpoint_dir
if not (cnn_checkpoint_dir / 'test_metrics.json').exists() and drive_cnn_checkpoint_dir is not None and (drive_cnn_checkpoint_dir / 'test_metrics.json').exists():
    cnn_checkpoint_dir = drive_cnn_checkpoint_dir

required_files = [
    wav_checkpoint_dir / 'test_metrics.json',
    wav_checkpoint_dir / 'val_metrics.json',
    wav_checkpoint_dir / 'test_classification_report.csv',
    cnn_checkpoint_dir / 'test_metrics.json',
    cnn_checkpoint_dir / 'val_metrics.json',
    cnn_checkpoint_dir / 'test_classification_report.csv',
]
missing = [str(path) for path in required_files if not path.exists()]
assert not missing, 'Missing comparison inputs: ' + ', '.join(missing)

print('wav2vec checkpoint:', wav_checkpoint_dir)
print('cnn checkpoint:', cnn_checkpoint_dir)
print('comparison output dir:', local_comparison_dir)
if drive_comparison_dir is not None:
    print('drive comparison dir:', drive_comparison_dir)


In [ ]:
from IPython.display import Markdown, display

import pandas as pd

from src.analysis.model_comparison import (
    build_comparison_markdown,
    build_per_class_f1_comparison_frame,
    build_split_metrics_frame,
    build_test_improvement_frame,
    load_checkpoint_metrics,
)

wav_metrics = load_checkpoint_metrics(wav_checkpoint_dir)
cnn_metrics = load_checkpoint_metrics(cnn_checkpoint_dir)

split_metrics_df = build_split_metrics_frame({
    'wav2vec2': wav_metrics,
    'cnn_baseline': cnn_metrics,
})
improvement_df = build_test_improvement_frame(
    baseline=cnn_metrics,
    main_model=wav_metrics,
    baseline_name='cnn_baseline',
    main_name='wav2vec2',
)
per_class_f1_df = build_per_class_f1_comparison_frame(
    baseline=cnn_metrics,
    main_model=wav_metrics,
    baseline_name='cnn_baseline',
    main_name='wav2vec2',
)
summary_markdown = build_comparison_markdown(
    baseline=cnn_metrics,
    main_model=wav_metrics,
    baseline_name='cnn_baseline',
    main_name='wav2vec2',
)

split_metrics_df.to_csv(local_comparison_dir / 'split_metrics_comparison.csv', index=False)
improvement_df.to_csv(local_comparison_dir / 'test_metric_gains.csv', index=False)
per_class_f1_df.to_csv(local_comparison_dir / 'per_class_f1_comparison.csv', index=False)
(local_comparison_dir / 'model_comparison_summary.md').write_text(summary_markdown, encoding='utf-8')

display(split_metrics_df.round(4))
display(improvement_df.round(4))
display(per_class_f1_df.round(4))
display(Markdown(summary_markdown))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.barplot(
    data=split_metrics_df[split_metrics_df['split'] == 'test'],
    x='model',
    y='macro_f1',
)
plt.title('Test Macro F1: wav2vec2 vs CNN Baseline')
plt.ylabel('Macro F1')
plt.xlabel('Model')
plt.tight_layout()
plt.savefig(local_comparison_dir / 'test_macro_f1_comparison.png', dpi=200)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plot_df = per_class_f1_df.melt(
    id_vars=['label'],
    value_vars=['cnn_baseline', 'wav2vec2'],
    var_name='model',
    value_name='f1_score',
)

plt.figure(figsize=(10, 5))
sns.barplot(data=plot_df, x='label', y='f1_score', hue='model')
plt.title('Per-Class Test F1 Comparison')
plt.xlabel('Emotion label')
plt.ylabel('F1 score')
plt.tight_layout()
plt.savefig(local_comparison_dir / 'per_class_f1_comparison.png', dpi=200)
plt.show()


In [ ]:
print('Comparison report directory:', local_comparison_dir)
for path in sorted(local_comparison_dir.iterdir()):
    print(path.name)


In [ ]:
if IS_COLAB:
    import shutil

    assert drive_comparison_dir is not None
    if drive_comparison_dir.exists():
        shutil.rmtree(drive_comparison_dir)
    shutil.copytree(local_comparison_dir, drive_comparison_dir)

    print('Backed up comparison artifacts to Google Drive:')
    print(' -', drive_comparison_dir)
    print('\nDrive comparison contents:')
    for path in sorted(drive_comparison_dir.iterdir()):
        kind = 'dir' if path.is_dir() else 'file'
        print(f' [{kind}] {path.name}')
else:
    print('Google Drive backup runs only in Colab.')
